# Age rank analysis  
All analysis related to age_rank. The files are created in the R scripts: salmon_map_dominance_consistent_script.R and create_full_annotation_fixed.R.


Imports

In [45]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path

pio.renderers.default = "notebook"

AGE_LABEL_MAP = {
    1: "(1)", 2: "(2)", 3: "(3)",
    4: "(4)", 5: "(5)", 6: "(6)",
    7: "(7)", 8: "(8)", 9: "(9) (C_mac)"
}


Load datasets

In [63]:
DATA_DIR = Path("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis")

# full_annotation has structural, eggNOG, orthofinder and age columns.
# Age is now assigned from gene tree traversal (parse_gene_trees_birth_nodes.py)
# rather than from Duplications.tsv heuristics.
full_annotation = pd.read_csv(DATA_DIR / "C_mac_full_annotation_with_age_fixed.csv")

# dup_long is the raw OrthoFinder duplication table expanded to one row per
# transcript per event. Each transcript appears many times here because it
# shows up in every ancestral duplication event, not just its birth event.
# Kept here to visualize the "all events" distribution for comparison with
# the one-birth-per-transcript approach used in all other plots.
dup_long = pd.read_csv(DATA_DIR / "C_mac_dup_long_all_events_May27.csv")

# prefilter_df: salmon-mapped results before expression filtering,
# annotated with full_annotation. 
prefilter_df = pd.read_csv(DATA_DIR / "prefilter_transcripts_annotated_May27.csv")

# de_results: post DE-analysis results, annotated with age_rank.
de_results = pd.read_csv(
    DATA_DIR / "salmon_map_dominance_DE_sex_results_new_filtering_genotype_controlled_age_rank_fixed.csv",
    float_precision="legacy"
)
de_results


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,eggNOG_OGs,species_tree_birth_node,gene_tree_birth_node,birth_type,age_rank,age_category,node_depth_from_root,branch_length,n_copies_in_og,n_species_in_og
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,A,...,"2FBM0@1|root,2TCUK@2759|Eukaryota,398E5@33154|...",C_maculatus_filtered_proteinfasta_TE_filtered,n3,duplication,9.0,recent,0.758700,0.061748,2.0,8.0
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,A,...,"KOG2101@1|root,KOG2101@2759|Eukaryota,396Y9@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n9,duplication,9.0,recent,0.758700,0.061748,2.0,14.0
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,A,...,"COG0181@1|root,KOG2892@2759|Eukaryota,38D6W@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n13,duplication,9.0,recent,0.758700,0.061748,2.0,14.0
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,A,...,"KOG2761@1|root,KOG2761@2759|Eukaryota,38F7R@33...",C_maculatus_filtered_proteinfasta_TE_filtered,n8,duplication,9.0,recent,0.758700,0.061748,2.0,12.0
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,A,...,"2E6V3@1|root,2SDHR@2759|Eukaryota",N8,NaN,mrca_inferred,6.0,intermediate,0.629385,0.171468,1.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,U,...,"2D3G7@1|root,2SRFN@2759|Eukaryota,3AMW9@33154|...",C_maculatus_filtered_proteinfasta_TE_filtered,n17,duplication,9.0,recent,0.758700,0.061748,23.0,4.0
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,U,...,NaN,C_maculatus_filtered_proteinfasta_TE_filtered,n43,duplication,9.0,recent,0.758700,0.061748,22.0,6.0
17572,19.121063,2.284212,0.326813,6.989347,2.761695e-12,6.049015e-12,g35167.t1,g35167,utg003885l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Duplications only, no mrca_inferred

In [86]:
# Filter to duplication birth type only for paralog-specific plots
de_results_dup = de_results[de_results["birth_type"] == "duplication"].copy()
sig_dup = de_results_dup[
    (de_results_dup["padj"] < 0.05) & 
    (de_results_dup["log2FoldChange"].abs() > 1)
].copy()

print(f"Expressed transcripts (duplication only): {len(de_results_dup):,}")
print(f"Sig DE transcripts (duplication only):    {len(sig_dup):,}")
print(f"\nFor comparison:")
print(f"Expressed transcripts (all birth types):  {len(de_results[de_results['age_rank'].notna()]):,}")
print(f"Sig DE transcripts (all birth types):     {len(sig[sig['age_rank'].notna()]):,}")

Expressed transcripts (duplication only): 9,585
Sig DE transcripts (duplication only):    4,207

For comparison:
Expressed transcripts (all birth types):  16,048
Sig DE transcripts (all birth types):     6,480


Color schemes and plot layout

In [64]:
# one  color for each dataset level
DATASET_COLORS = {
    "full_annotation": "#9E9E9E",   # grey         — genome-wide
    "prefilter_df":    "#C4A35A",   # muted gold   — detectable
    "de_results":      "#5BA67A",   # medium green — expressed
    "sig":             "#7DD181",   # bright green — significantly DE
}

DATASET_LABELS = {
    "full_annotation": "Full Genome Annotation",
    "prefilter_df":    "Salmon-Mapped (pre-filter)",
    "de_results":      "Expressed (≥5 counts in ≥5 samples)",
    "sig":             "Significantly DE (padj < 0.05, |LFC| > 1)",
}

LAYOUT_BASE = dict(
    plot_bgcolor="white",
    font=dict(family="Arial", size=12),
    xaxis=dict(
        title="Age Rank (1 = oldest at root, 9 = youngest at tip)",
        showgrid=False,
        categoryorder="array",
        categoryarray=[AGE_LABEL_MAP[i] for i in range(1, 10)]
    ),
    yaxis=dict(
        title="Number of Transcripts",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
    ),
    legend=dict(
        title="Dataset",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right",  x=1
    ),
    bargap=0.2,
)

All duplication events plot. From dup_long (includes ALL events a transcript appears in, not just its birth event). One transcript can appear many times here because OrthoFinder lists it as a descendant of every ancestral duplication in its history. Compare this to fig2 below where each transcript gets exactly one birth event from gene tree traversal.

In [65]:
fig1 = go.Figure()

dup_age_data = (
    dup_long[dup_long["age_rank"].notna()]
    .assign(
        age_rank  = lambda x: x["age_rank"].astype(int),
        age_label = lambda x: x["age_rank"].astype(int).map(AGE_LABEL_MAP),
    )
    .groupby(["age_rank", "age_label"])
    .size().reset_index(name="count")
    .sort_values("age_rank")
)

fig1.add_trace(go.Bar(
    x=dup_age_data["age_label"],
    y=dup_age_data["count"],
    marker_color=DATASET_COLORS["full_annotation"],
    marker_line_color="black",
    marker_line_width=1,
    text=dup_age_data["count"],
    textposition="outside",
    textfont=dict(size=11, color="black", family="Arial Black"),
    showlegend=False,
))

fig1.update_layout(
    **LAYOUT_BASE,
    width=1000,
    height=700,
    title=dict(
        text=(
            f"<b>All Duplication Events in <i>C. maculatus</i> Lineage</b>"
            f"<br><sup>Based on {len(dup_long):,} total events - "
            f"one transcript can appear multiple times (nested duplication history)</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    yaxis_title="Number of Duplication Events",
    showlegend=False,
)
fig1.write_image("age_rank_all_dup_events_distribution_.svg")
fig1.show()

Combined plot of the 4 datasets (all transcripts in each that have age_rank information):  
Full annotation - all C_mac transcripts in the orthofinder annotation  
Prefilter - Expressed and unexpressed trancripts. All that were mapped by salmon    
DE-results - All expressed transcripts (5 counts in 5 samples)  
Sig - Only significantly DE transcripts (M v F)  

Transcripts at low ranks are not assigned an age rank instead of NA trhough MRCA based on where it had its last common anscestor

In [67]:
sig = de_results[(de_results["padj"] < 0.05) & (de_results["log2FoldChange"].abs() > 1)]

datasets_combined = {
    "full_annotation": full_annotation,
    "prefilter_df":    prefilter_df,
    "de_results":      de_results,
    "sig":             sig,
}

all_labels = [AGE_LABEL_MAP[i] for i in range(1, 10)]

fig2 = go.Figure()

for ds_key, df in datasets_combined.items():
    age_data = (
        df[df["age_rank"].notna()]
        .assign(
            age_rank  = lambda x: x["age_rank"].astype(int),
            age_label = lambda x: x["age_rank"].astype(int).map(AGE_LABEL_MAP),
        )
        .groupby(["age_rank", "age_label"])
        .size().reset_index(name="count")
        .sort_values("age_rank")
    )

    # Align to all 9 positions
    counts, texts = [], []
    for rank in range(1, 10):
        row = age_data[age_data["age_rank"] == rank]
        if len(row) > 0:
            counts.append(row.iloc[0]["count"])
            texts.append(str(row.iloc[0]["count"]))
        else:
            counts.append(0)
            texts.append("")

    fig2.add_trace(go.Bar(
        x=all_labels,
        y=counts,
        name=DATASET_LABELS[ds_key],
        marker_color=DATASET_COLORS[ds_key],
        marker_line_color="black",
        marker_line_width=0.8,
        text=texts,
        textposition="outside",
        textfont=dict(size=9, color="black"),
    ))

fig2.update_layout(
    **LAYOUT_BASE,
    barmode="group",
    bargroupgap=0.08,
    width=1200,
    title=dict(
        text=(
            "<b>Age Rank Distribution Across Dataset Levels</b>"
            "<br><sup>grey = genome-wide · gold = detectable · "
            "medium green = expressed · bright green = significantly DE</sup>"
        ),
        x=0.5, xanchor="center"
    ),
)
fig2.write_image("age_rank_dup_distribution_.svg")
fig2.show()

# nr of transcripts in each step:
for name, df in [("full_annotation", full_annotation),
                 ("prefilter_df",    prefilter_df),
                 ("de_results",      de_results),
                 ("significant_de",  sig)]:

    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]} cols")
    print(f"  age_rank coverage: {df['age_rank'].notna().sum():,} transcripts with age info\n")


full_annotation: 37,988 rows × 28 cols
  age_rank coverage: 28,966 transcripts with age info

prefilter_df: 36,382 rows × 28 cols
  age_rank coverage: 27,926 transcripts with age info

de_results: 17,574 rows × 34 cols
  age_rank coverage: 16,048 transcripts with age info

significant_de: 7,270 rows × 34 cols
  age_rank coverage: 6,480 transcripts with age info



We compare this to five related species to see if C_mac stands out with this pattern. 

In [68]:
# ── Load species annotations (same DATA_DIR as existing notebook) ─────────────
species_annotations = {
    "C. septempunctata": pd.read_csv(DATA_DIR / "C_septempunctata_full_annotation_with_age_fixed.csv"),
    "T. castaneum":      pd.read_csv(DATA_DIR / "T_castaneum_full_annotation_with_age_fixed.csv"),
    "A. obtectus":       pd.read_csv(DATA_DIR / "A_obtectus_full_annotation_with_age_fixed.csv"),
    "B. siliquastri":    pd.read_csv(DATA_DIR / "B_siliquastri_full_annotation_with_age_fixed.csv"),
    "C. chinensis":      pd.read_csv(DATA_DIR / "C_chinensis_full_annotation_with_age_fixed.csv"),
    "C. maculatus":      pd.read_csv(DATA_DIR / "C_mac_full_annotation_with_age_fixed.csv"),
}

# Node names from R lineage output - rank 1 = oldest (root), max rank = species tip
# The species have different number of nodes pased on phylogeny 
SPECIES_AGE_LABEL_MAP = {
    "C. septempunctata": {1:"N0", 2:"N1", 3:"N3", 4:"C. sep"},
    "T. castaneum":      {1:"N0", 2:"N1", 3:"N3", 4:"N4", 5:"N5", 6:"N7",  7:"T. cas"},
    "A. obtectus":       {1:"N0", 2:"N1", 3:"N3", 4:"N4", 5:"N6", 6:"N8", 7:"A. obt"},
    "B. siliquastri":    {1:"N0", 2:"N1", 3:"N3", 4:"N4", 5:"N6", 6:"N8", 7:"N11", 8:"B. sil"},
    "C. chinensis":      {1:"N0", 2:"N1", 3:"N3", 4:"N4", 5:"N6", 6:"N8", 7:"N11", 8:"N12", 9:"C. chi"},
    "C. maculatus":      {1:"N0", 2:"N1", 3:"N3", 4:"N4", 5:"N6", 6:"N8", 7:"N11", 8:"N12", 9:"C. mac"},
}

SPECIES_COLORS = {
    "C. septempunctata": "#E8A598",   # dusty rose
    "T. castaneum":      "#E8C078",   # soft amber
    "A. obtectus":       "#8DB8A0",   # sage green
    "B. siliquastri":    "#7EB5C8",   # dusty blue
    "C. chinensis":      "#B0A4D0",   # soft lavender
    "C. maculatus":      "#A8A8A8",   # soft grey
}

# ── Subplots: 3 rows x 2 cols ─────────────────────────────────────────────────
subplot_titles = [
    f"{species} ({df['age_rank'].notna().sum():,} total)"
    for species, df in species_annotations.items()
]

fig_species = make_subplots(
    rows=3, cols=2,
    subplot_titles=subplot_titles,
    vertical_spacing=0.12,
    horizontal_spacing=0.10,
)

positions = [(1,1),(1,2),(2,1),(2,2),(3,1),(3,2)]

for (species, df), (row, col) in zip(species_annotations.items(), positions):

    label_map  = SPECIES_AGE_LABEL_MAP[species]
    max_rank   = max(label_map.keys())
    all_labels = [label_map[i] for i in range(1, max_rank + 1)]

    counts = (
        df[df["age_rank"].notna()]
        .assign(age_rank=lambda x: x["age_rank"].astype(int))
        .groupby("age_rank")
        .size()
        .reindex(range(1, max_rank + 1), fill_value=0)
        .tolist()
    )
    texts = [str(c) if c > 0 else "" for c in counts]

    fig_species.add_trace(
        go.Bar(
            x=all_labels,
            y=counts,
            name=species,
            marker_color=SPECIES_COLORS[species],
            marker_line_color="black",
            marker_line_width=0.8,
            text=texts,
            textposition="outside",
            textfont=dict(size=9, color="black"),
            showlegend=False,
        ),
        row=row, col=col,
    )

    fig_species.update_xaxes(
        title_text="Age Rank (1 = oldest at root)",
        showgrid=False,
        categoryorder="array",
        categoryarray=all_labels,
        showline=True, linecolor="black", mirror=True, ticks="outside",
        row=row, col=col,
    )
    fig_species.update_yaxes(
        title_text="Number of Transcripts",
        showgrid=True, gridcolor="lightgrey", zeroline=False,
        showline=True, linecolor="black", mirror=True, ticks="outside",
        range=[0, max(counts) * 1.10],
        row=row, col=col,
    )

fig_species.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    height=1000,
    width=1000,
    bargap=0.2,
    title=dict(
        text=(
            "<b>Age Rank Distribution - Genome-wide Annotation</b>"
        ),
        x=0.5, xanchor="center",
    ),
)
fig_species.write_image("age_rank_species_comparison.svg")
fig_species.show()

# ── Summary ───────────────────────────────────────────────────────────────────
for species, df in species_annotations.items():
    print(f"{species}: {df.shape[0]:,} total transcripts  |  "
          f"{df['age_rank'].notna().sum():,} with age_rank assigned")

C. septempunctata: 19,849 total transcripts  |  17,214 with age_rank assigned
T. castaneum: 17,597 total transcripts  |  14,024 with age_rank assigned
A. obtectus: 29,838 total transcripts  |  25,609 with age_rank assigned
B. siliquastri: 14,999 total transcripts  |  13,263 with age_rank assigned
C. chinensis: 25,807 total transcripts  |  21,482 with age_rank assigned
C. maculatus: 37,988 total transcripts  |  28,966 with age_rank assigned


Split by log2FC sign. log2FC > 1 = male-biased, log2FC <1 = female-biased. One box plot for each age rank 

In [70]:
sig = de_results[(de_results["padj"] < 0.05) & (de_results["log2FoldChange"].abs() > 1)].copy()

sig["abs_lfc"]   = sig["log2FoldChange"].abs()
sig["sex_bias"]  = sig["log2FoldChange"].apply(lambda x: "Male-biased" if x > 0 else "Female-biased")
sig["age_label"] = sig["age_rank"].map(AGE_LABEL_MAP)

sig_age = sig[sig["age_rank"].notna()].copy()
sig_age["age_rank"] = sig_age["age_rank"].astype(int)

all_labels = [AGE_LABEL_MAP[i] for i in range(1, 10)]

#Missing transcripts from the volcano plot: 
sig_total   = de_results[(de_results["padj"] < 0.05) & (de_results["log2FoldChange"].abs() > 1)]
sig_with_age = sig_total[sig_total["age_rank"].notna()]

male_total   = (sig_total["log2FoldChange"] > 0).sum()
female_total = (sig_total["log2FoldChange"] < 0).sum()

male_age   = (sig_with_age["log2FoldChange"] > 0).sum()
female_age = (sig_with_age["log2FoldChange"] < 0).sum()

print(f"All sig DE:         {len(sig_total):,}  (M: {male_total:,}, F: {female_total:,})")
print(f"With age rank:      {len(sig_with_age):,}  (M: {male_age:,}, F: {female_age:,})")

# Out of the missing ones:
# No HOG at all = gene absent from OrthoFinder entirely (too diverged,
#   too short, or a TE-derived gene OrthoFinder could not cluster)
# Have a HOG but no age_rank = birth_type is species_specific (no orthologs
#   detected in any other species, age genuinely unknown) OR the gene was
#   not present in any resolved gene tree (absent from transcript_birth_nodes.tsv)

sig_no_age = sig_total[sig_total["age_rank"].isna()]

has_hog    = sig_no_age["HOG"].notna().sum()
no_hog     = sig_no_age["HOG"].isna().sum()

# Birth type breakdown for transcripts that do have an age
print("\nBirth type breakdown (sig transcripts with age_rank):")
print(sig_total[sig_total["age_rank"].notna()]["birth_type"].value_counts())

print(f"Missing age rank:         {len(sig_no_age):,}")
print(f"  Have a HOG (in family): {has_hog:,}")
print(f"  No HOG at all:          {no_hog:,}")

SEX_COLORS = {
    "Male-biased":   "#6BAED6",
    "Female-biased": "#E07B8A",
}

fig = go.Figure()

for bias, color in SEX_COLORS.items():
    sub = sig_age[sig_age["sex_bias"] == bias]
    counts = sub.groupby("age_rank").size().reset_index(name="n")

    fig.add_trace(go.Box(
        x=sub["age_label"],
        y=sub["abs_lfc"],
        name=bias,
        fillcolor=color,               # fills the box
        marker=dict(
            color=color,               # outlier dot color
            size=4,
            opacity=0.7,
        ),
        line=dict(color="black", width=0.8),
        boxmean=True,
        offsetgroup=bias,
        customdata=sub[["transcript_id", "HOG"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "HOG: %{customdata[1]}<br>"
            "abs(log2FC): %{y:.3f}<extra></extra>"
        ),
    ))

    # n label below each box, above the x-axis tick
    for _, row in counts.iterrows():
        label = AGE_LABEL_MAP[int(row["age_rank"])]
        fig.add_annotation(
            x=label,
            y=0,
            text=f"({int(row['n'])})",
            showarrow=False,
            yref="y",
            yshift=-10,
            font=dict(size=9, color=color),
            xshift=-21 if bias == "Male-biased" else 20,
        )

fig.update_layout(
    plot_bgcolor="white",
    boxmode="group",
    height=600,
    width=1400,
    font=dict(family="Arial", size=12),
    title=dict(
        text=(
            "<b>Absolute Log2 Fold Change by Age Rank - Significantly DE Transcripts M vs. F</b>"
            "<br><sup>padj < 0.05, |log2FC| > 1 ... Dashed line inside box = mean</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Age Rank (1 = oldest, 9 = youngest)",
        showgrid=False,
        categoryorder="array",
        categoryarray=all_labels,
    ),
    yaxis=dict(
        title="abs(log2FoldChange)",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=True,
        zerolinecolor="lightgrey",
    ),
    legend=dict(
        title="Sex Bias:",
        orientation="h",
        yanchor="bottom", y=1.0,
        xanchor="right",  x=1,
    ),
)
fig.write_image("age_rank_sig_transcripts_boxplot.svg")
fig.show()

All sig DE:         7,270  (M: 5,070, F: 2,200)
With age rank:      6,480  (M: 4,409, F: 2,071)

Birth type breakdown (sig transcripts with age_rank):
birth_type
duplication      4207
mrca_inferred    2273
Name: count, dtype: int64
Missing age rank:         790
  Have a HOG (in family): 299
  No HOG at all:          491


Age ranks 5, 6, 7 interesting here. Something in the bruchid nodes leads to hihger male-bias. 

Note the proportion of significantly DE transcripts that lack age_rank.  
All sig DE:         7,270  (M: 5,070, F: 2,200)  
With age rank:      6,480  (M: 4,409, F: 2,071)  
These split into:
birth_type=species_specific (no orthologs outside C. mac, age genuinely unknown) and
transcripts absent from OrthoFinder entirely (too diverged, filtered, or lineage-specific).

Proportional bar plots

In [82]:
# Classify all expressed transcripts with age info
de_age = de_results[de_results["age_rank"].notna()].copy()
de_age["age_rank"] = de_age["age_rank"].astype(int)
de_age["age_label"] = de_age["age_rank"].map(AGE_LABEL_MAP)

def classify_bias(row):
    if row["padj"] < 0.05 and row["log2FoldChange"] > 1:
        return "Male-biased"
    elif row["padj"] < 0.05 and row["log2FoldChange"] < -1:
        return "Female-biased"
    else:
        return "Unbiased"

de_age["bias"] = de_age.apply(classify_bias, axis=1)

# Count and compute proportions per age rank
counts = (
    de_age.groupby(["age_rank", "age_label", "bias"])
    .size().reset_index(name="count")
)

totals = counts.groupby("age_rank")["count"].sum().reset_index(name="total")
counts = counts.merge(totals, on="age_rank")
counts["proportion"] = counts["count"] / counts["total"] * 100

all_labels = [AGE_LABEL_MAP[i] for i in range(1, 10)]

BIAS_COLORS = {
    "Unbiased":      "#708090",  # slate grey-blue
    "Male-biased":   "#6BAED6",  # same blue as boxplot
    "Female-biased": "#E07B8A",  # same red as boxplot
}

fig = go.Figure()

for bias, color in BIAS_COLORS.items():
    sub = counts[counts["bias"] == bias].sort_values("age_rank")

    # Align to all 9 positions
    props, texts, customdata = [], [], []
    for rank in range(1, 10):
        row = sub[sub["age_rank"] == rank]
        if len(row) > 0:
            p = row.iloc[0]["proportion"]
            n = row.iloc[0]["count"]
            props.append(p)
            texts.append(f"{p:.1f}%")
            customdata.append(n)
        else:
            props.append(0)
            texts.append("")
            customdata.append(0)

    fig.add_trace(go.Bar(
        x=all_labels,
        y=props,
        name=bias,
        marker_color=color,
        marker_line_color="black",
        marker_line_width=0.8,
        text=texts,
        textposition="inside",
        textfont=dict(size=10, color="white", family="Arial Black"),
        customdata=customdata,
        hovertemplate=(
            f"<b>{bias}</b><br>"
            "Age rank: %{x}<br>"
            "Proportion: %{y:.1f}%<br>"
            "n transcripts: %{customdata}<extra></extra>"
        ),
    ))

# Add total n below each bar
totals_by_rank = de_age.groupby(["age_rank"]).size().reset_index(name="n")
for _, row in totals_by_rank.iterrows():
    fig.add_annotation(
        x=AGE_LABEL_MAP[int(row["age_rank"])],
        y=0,
        yref="y",
        text=f"(n={int(row['n'])})",
        showarrow=False,
        yshift=-10,
        font=dict(size=10, color="grey"),
    )

fig.update_layout(
    plot_bgcolor="white",
    barmode="stack",
    font=dict(family="Arial", size=12),
    title=dict(
        text=(
            "<b>Proportion of Sex-Biased Transcripts by Age Rank</b>"
            "<br><sup> All expressed transcripts (≥5 counts in ≥5 samples) * "
            "Sex-bias defined as: padj < 0.05, |log2FC| > 1</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Age Rank (1 = oldest, 9 = youngest)",
        showgrid=False,
        categoryorder="array",
        categoryarray=all_labels,
        color="black",
        tickvals=all_labels,
        ticktext=[f"<br>{label}" for label in all_labels], 
    ),
    yaxis=dict(
        title="Proportion of Transcripts (%)",
        color="black",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
        range=[0, 105],
    ),
    legend=dict(
        title="Sex Bias",
        orientation="h",
        yanchor="bottom", y=0.98,
        xanchor="right", x=1,
    ),
    margin=dict(b=80),
)
fig.write_image("age_rank_sex_bias_proportions.svg")
fig.show()

Gene family size diversity. Combined plot 4 datasets (all transcripts in each that have age_rank and gene family information):  
Full annotation - all C_mac transcripts in the orthofinder annotation  
Prefilter - Expressed and unexpressed trancripts. All that were mapped by salmon    
DE-results - All expressed transcripts (5 counts in 5 samples)  
Sig - Only significantly DE transcripts (M v F)  

In [72]:
sig = de_results[(de_results["padj"] < 0.05) & (de_results["log2FoldChange"].abs() > 1)].copy()

datasets = {
    "full_annotation": ("Full Genome Annotation",               full_annotation, "#9E9E9E"),
    "prefilter_df":    ("Salmon-Mapped (pre-filter)",            prefilter_df,    "#C4A35A"),
    "de_results":      ("Expressed (≥5 counts in ≥5 samples)",  de_results,      "#5BA67A"),
    "sig":             ("Significantly DE (padj<0.05, |LFC|>1)", sig,             "#7DD181"),
}

def compute_hog_age_diversity(df):
    """Compute how many HOGs have each number of distinct age ranks."""
    return (
        df[df["age_rank"].notna() & df["HOG"].notna()]
        .assign(age_rank=lambda x: x["age_rank"].astype(int))
        .groupby("HOG")["age_rank"]
        .nunique()
        .reset_index(name="n_ages")
        .groupby("n_ages")
        .size()
        .reset_index(name="number_of_HOGs")
        .rename(columns={"n_ages": "number_of_distinct_ages"})
        .sort_values("number_of_distinct_ages")
    )

fig = go.Figure()

for ds_key, (ds_label, df, color) in datasets.items():
    diversity = compute_hog_age_diversity(df)

    fig.add_trace(go.Bar(
        x=diversity["number_of_distinct_ages"],
        y=diversity["number_of_HOGs"],
        name=ds_label,
        marker_color=color,
        marker_line_color="black",
        marker_line_width=0.8,
        text=diversity["number_of_HOGs"],
        textposition="outside",
        textfont=dict(size=9, color="black"),
        hovertemplate=(
            f"<b>{ds_label}</b><br>"
            "Distinct ages per gene family: %{x}<br>"
            "Number of gene families: %{y}<extra></extra>"
        ),
    ))

fig.update_layout(
    plot_bgcolor="white",
    barmode="group",
    bargroupgap=0.08,
    font=dict(family="Arial", size=12),
    title=dict(
        text=(
            "<b>Gene Family Age Diversity Across Dataset Levels</b>"
            "<br><sup>Number of gene families containing transcripts with N distinct duplication ages · "
            "Gene families with only 1 age = all transcripts duplicated at same node</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Number of Distinct Ages within Gene Family",
        showgrid=False,
        tickmode="linear",
        dtick=1,
    ),
    yaxis=dict(
        title="Number of Gene families",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
    ),
    legend=dict(
        title="Dataset",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
)
fig.show()

# transcript counts at each filtering stage for HOG age diversity
for name, df in [("full_annotation", full_annotation),
                 ("prefilter_df",    prefilter_df),
                 ("de_results",      de_results),
                 ("sig",             sig)]:

    total        = len(df)
    has_age      = df["age_rank"].notna().sum()
    has_hog      = df["HOG"].notna().sum()
    has_both     = df[df["age_rank"].notna() & df["HOG"].notna()].shape[0]

    print(f"{name}: {total:,} total transcripts")
    print(f"  with age_rank:       {has_age:,}  ({100*has_age/total:.1f}%)")
    print(f"  with HOG:            {has_hog:,}  ({100*has_hog/total:.1f}%)")
    print(f"  with both (plotted): {has_both:,}  ({100*has_both/total:.1f}%)")
    print()


full_annotation: 37,988 total transcripts
  with age_rank:       28,966  (76.3%)
  with HOG:            31,408  (82.7%)
  with both (plotted): 26,729  (70.4%)

prefilter_df: 36,382 total transcripts
  with age_rank:       27,926  (76.8%)
  with HOG:            29,943  (82.3%)
  with both (plotted): 25,767  (70.8%)

de_results: 17,574 total transcripts
  with age_rank:       16,048  (91.3%)
  with HOG:            15,594  (88.7%)
  with both (plotted): 15,011  (85.4%)

sig: 7,270 total transcripts
  with age_rank:       6,480  (89.1%)
  with HOG:            6,450  (88.7%)
  with both (plotted): 6,151  (84.6%)



Top 20 most age diverse gene families of significantlly DE transcripts

In [73]:
sig_hog_diversity = (
    sig[sig["age_rank"].notna() & sig["HOG"].notna()]
    .assign(age_rank=lambda x: x["age_rank"].astype(int))
    .groupby("HOG")
    .agg(
        n_transcripts=("transcript_id", "count"),
        n_ages=("age_rank", "nunique"),
        min_age=("age_rank", "min"),
        max_age=("age_rank", "max"),
    )
    .assign(age_range=lambda x: x["max_age"] - x["min_age"] + 1)
    .query("n_ages > 1")  # only HOGs with mixed ages
    .sort_values(["n_ages", "n_transcripts"], ascending=[False, False])
    .head(10)
    .reset_index()
)

# Sort ascending for horizontal bar chart (highest at top)
sig_hog_diversity = sig_hog_diversity.sort_values("n_ages", ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    x=sig_hog_diversity["n_ages"],
    y=sig_hog_diversity["HOG"],
    orientation="h",
    marker=dict(
        color=sig_hog_diversity["age_range"],
        colorscale=[[0, "#96E8BC"], [1, "#2E6B47"]],
        colorbar=dict(
            title="Age Range",
            thickness=15,
            len=0.6,
        ),
        line=dict(color="black", width=0.8),
    ),
    text=[f"n={n}" for n in sig_hog_diversity["n_transcripts"]],
    textposition="outside",
    textfont=dict(size=10, color="black"),
    customdata=sig_hog_diversity[["n_transcripts", "age_range", "min_age", "max_age"]].values,
    hovertemplate=(
        "<b>Gene Family: %{y}</b><br>"
        "Distinct ages: %{x}<br>"
        "Age range: %{customdata[1]} (rank %{customdata[2]}–%{customdata[3]})<br>"
        "Total sig DE transcripts: %{customdata[0]}<extra></extra>"
    ),
))

fig.update_layout(
    plot_bgcolor="white",
    font=dict(family="Arial", size=12),
    title=dict(
        text=(
            "<b>Top 10 Most Age-Diverse Gene Families - Significantly DE Transcripts</b>"
            "<br><sup>padj < 0.05, |LFC| > 1 · bar labels show total sig DE transcripts per gene family · "
            "color = age range (light = narrow, dark = broad)</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Number of Distinct Ages per Gene Family",
        showgrid=True,
        gridcolor="lightgrey",
        tickmode="linear",
        dtick=1,
        range=[0, sig_hog_diversity["n_ages"].max() + 1.5],  # room for text labels
    ),
    yaxis=dict(
        title="Gene Family (HOG)",
        showgrid=False,
        automargin=True,
    ),
    margin=dict(r=100),
    height=600,
)
fig.show()

# Age rank and gene family size

Explore the question: does gene family size vary with evolutionary age, does it interact with sex bias? 

Set up dataset. Add columns for the genome wide gene family sizes and the mapped gene family sizes.  
Add sex bias column.   
Subset only significantly DE transcripts

In [74]:
# Compute sizes from each level independently
hog_size_genome    = (full_annotation.dropna(subset=["HOG"])
                      .groupby("HOG").size()
                      .reset_index(name="hog_size_genome"))

hog_size_mapped    = (prefilter_df.dropna(subset=["HOG"])
                      .groupby("HOG").size()
                      .reset_index(name="hog_size_mapped"))

hog_size_expressed = (de_results.dropna(subset=["HOG"])
                      .groupby("HOG").size()
                      .reset_index(name="hog_size_expressed"))

# Merge all three onto de_results
de_results_hog = (de_results
                  .merge(hog_size_genome,    on="HOG", how="left")
                  .merge(hog_size_mapped,    on="HOG", how="left")
                  .merge(hog_size_expressed, on="HOG", how="left"))

# Add bias classification
def classify_bias(row):
    if row["padj"] < 0.05 and row["log2FoldChange"] > 1:
        return "Male-biased"
    elif row["padj"] < 0.05 and row["log2FoldChange"] < -1:
        return "Female-biased"
    else:
        return "Unbiased"

de_results_hog["bias"] = de_results_hog.apply(classify_bias, axis=1)

# Sig only subset
sig_hog = de_results_hog[
    (de_results_hog["padj"] < 0.05) &
    (de_results_hog["log2FoldChange"].abs() > 1)
].copy()

# Sanity check
print("de_results_hog:", de_results_hog.shape)
print("sig_hog:        ", sig_hog.shape)
print()
for col in ["hog_size_genome", "hog_size_mapped", "hog_size_expressed"]:
    n = de_results_hog[col].notna().sum()
    print(f"  {col}: {n:,} transcripts with gene family size info")

has_both = de_results_hog[
    de_results_hog["age_rank"].notna() &
    de_results_hog["hog_size_genome"].notna()
]
has_both_sig = sig_hog[
    sig_hog["age_rank"].notna() &
    sig_hog["hog_size_genome"].notna()
]

print(f"\n de_results with age_rank + gene family: {len(has_both):,}")
print(f"sig with age_rank + gene family:        {len(has_both_sig):,}")
de_results_hog

de_results_hog: (17574, 38)
sig_hog:         (7270, 38)

  hog_size_genome: 15,594 transcripts with gene family size info
  hog_size_mapped: 15,594 transcripts with gene family size info
  hog_size_expressed: 15,594 transcripts with gene family size info

 de_results with age_rank + gene family: 15,011
sig with age_rank + gene family:        6,151


,baseMean,log2FoldChange,lfcSE,stat,pvalue,padj,transcript_id,gene_id,seqname,chr_loc,...,age_rank,age_category,node_depth_from_root,branch_length,n_copies_in_og,n_species_in_og,hog_size_genome,hog_size_mapped,hog_size_expressed,bias
0,248.420188,0.640709,0.110526,5.796904,6.755020e-09,1.316673e-08,g2.t1,g2,utg000001l,A,...,9.0,recent,0.758700,0.061748,2.0,8.0,2.0,2.0,2.0,Unbiased
1,92.726112,0.650988,0.104454,6.232321,4.595741e-10,9.353175e-10,g3.t1,g3,utg000001l,A,...,9.0,recent,0.758700,0.061748,2.0,14.0,2.0,2.0,2.0,Unbiased
2,136.044931,-0.600974,0.070022,-8.582668,9.269779e-18,2.387530e-17,g4.t1,g4,utg000001l,A,...,9.0,recent,0.758700,0.061748,2.0,14.0,2.0,2.0,2.0,Unbiased
3,3.332468,2.699629,0.718636,3.756600,1.722376e-04,2.671748e-04,g5.t1,g5,utg000001l,A,...,9.0,recent,0.758700,0.061748,2.0,12.0,2.0,2.0,2.0,Male-biased
4,381.084979,-0.875080,0.046175,-18.951587,4.284679e-80,3.659726e-79,g6.t1,g6,utg000001l,A,...,6.0,intermediate,0.629385,0.171468,1.0,4.0,1.0,1.0,1.0,Unbiased
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17569,20.275098,0.319252,0.150056,2.127555,3.337403e-02,4.238130e-02,g34843.t1,g34843,utg003648l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,2.0,2.0,2.0,Unbiased
17570,1.364118,2.068769,0.590743,3.501981,4.618132e-04,6.956768e-04,g34884.t1,g34884,utg003700l,U,...,9.0,recent,0.758700,0.061748,23.0,4.0,23.0,23.0,9.0,Male-biased
17571,8.286716,1.984267,0.355927,5.574921,2.476423e-08,4.718367e-08,g34922.t1,g34922,utg003714l,U,...,9.0,recent,0.758700,0.061748,22.0,6.0,22.0,21.0,8.0,Male-biased
17572,19.121063,2.284212,0.326813,6.989347,2.761695e-12,6.049015e-12,g35167.t1,g35167,utg003885l,U,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Male-biased


Boxplot - gene family size (log) vs age rank

In [85]:
box_df = de_results_hog[
    de_results_hog["age_rank"].notna() &
    de_results_hog["hog_size_genome"].notna()
].copy()
box_df["age_rank"]  = box_df["age_rank"].astype(int)
box_df["age_label"] = box_df["age_rank"].map(AGE_LABEL_MAP)

all_labels = [AGE_LABEL_MAP[i] for i in range(1, 10)]

fig = go.Figure()

for bias in ["Unbiased", "Male-biased", "Female-biased"]:
    sub    = box_df[box_df["bias"] == bias]
    counts = sub.groupby("age_rank").size().reset_index(name="n")

    fig.add_trace(go.Box(
        x=sub["age_label"],
        y=sub["hog_size_genome"],
        name=bias,
        fillcolor=BIAS_COLORS[bias],
        marker=dict(color=BIAS_COLORS[bias], size=3, opacity=0.5),
        line=dict(color="black", width=0.8),
        boxmean=True,
        offsetgroup=bias,
        customdata=sub[["transcript_id", "HOG"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Gene family: %{customdata[1]}<br>"
            "Age rank: %{x}<br>"
            "Family size (genome): %{y}<extra></extra>"
        ),
    ))

    # (n) labels below each box
    for _, row in counts.iterrows():
        label = AGE_LABEL_MAP[int(row["age_rank"])]
        fig.add_annotation(
            x=label,
            y=0,
            text=f"({int(row['n'])})",
            showarrow=False,
            yshift=-15,
            font=dict(size=9, color=BIAS_COLORS[bias]),
            xshift=-28 if bias == "Unbiased" else (0 if bias == "Male-biased" else 28),
            textangle =-45
        )

fig.update_layout(
    plot_bgcolor="white",
    boxmode="group",
    font=dict(family="Arial", size=12),
    title=dict(
        text=(
            "<b>Gene family size by age rank - Expressed transcripts</b>"
            "<br><sup>Gene family size defined at genome level · "
            "dashed line = mean · n = transcripts per group</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Age rank (1 = oldest, 9 = youngest)",
        showgrid=False,
        categoryorder="array",
        categoryarray=all_labels,
    ),
    yaxis=dict(
        title="Gene family size (genome, log-scale)",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
        #type="log",
        tickformat=".0f"
    ),
    legend=dict(
        title="Sex bias",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    margin=dict(b=80),
)
fig.show()

At the oldest rank are mainly singelton transcripts (the most recent copy was created here). If it had following duplications those were selected away. Larger gene family sizes are more common in C_mac which makes sense as its genome is so vast, repetetive and contains so many genes. 

In [80]:
prop_df2 = de_results_hog[
    de_results_hog["age_rank"].notna() &
    de_results_hog["hog_size_genome"].notna()
].copy()
prop_df2["age_rank"]  = prop_df2["age_rank"].astype(int)
prop_df2["age_label"] = prop_df2["age_rank"].map(AGE_LABEL_MAP)

AGE_RANK_COLORS = {
    1: "#8B4513", 2: "#A0522D", 3: "#CD853F",
    4: "#DAA520", 5: "#BDB76B", 6: "#6B8E23",
    7: "#3CB371", 8: "#2E8B57", 9: "#228B22",
}

counts2 = (
    prop_df2.groupby(["hog_size_genome", "age_rank", "age_label"])
    .size().reset_index(name="count")
)
totals2 = counts2.groupby("hog_size_genome")["count"].sum().reset_index(name="total")
counts2 = counts2.merge(totals2, on="hog_size_genome")
counts2["pct"] = counts2["count"] / counts2["total"] * 100

all_sizes = sorted(prop_df2["hog_size_genome"].unique())

fig = go.Figure()

for rank in range(1, 10):
    sub = counts2[counts2["age_rank"] == rank].sort_values("hog_size_genome")

    props, texts, customdata = [], [], []
    for size in all_sizes:
        row = sub[sub["hog_size_genome"] == size]
        if len(row) > 0:
            p = row.iloc[0]["pct"]
            n = row.iloc[0]["count"]
            t = row.iloc[0]["total"]
            props.append(p)
            texts.append(f"{p:.1f}%" if p >= 8 else "")
            customdata.append([n, t])
        else:
            props.append(0)
            texts.append("")
            customdata.append([0, 0])

    fig.add_trace(go.Bar(
        x=all_sizes,
        y=props,
        name=AGE_LABEL_MAP[rank],
        marker_color=AGE_RANK_COLORS[rank],
        marker_line_color="black",
        marker_line_width=0.5,
        text=texts,
        textposition="inside",
        textfont=dict(size=9, color="white"),
        customdata=customdata,
        hovertemplate=(
            f"<b>Age rank: {AGE_LABEL_MAP[rank]}</b><br>"
            "Gene family size: %{x}<br>"
            "Proportion: %{y:.1f}%<br>"
            "Count: %{customdata[0]}<br>"
            "Total at this size: %{customdata[1]}<extra></extra>"
        ),
    ))

# Total n annotations
for size in all_sizes:
    total = counts2[counts2["hog_size_genome"] == size]["total"].iloc[0]
    fig.add_annotation(
        x=size,
        y=1,
        yref="paper",
        text=f"(n={total})",
        showarrow=False,
        yshift=-35,
        font=dict(size=9, color="black"),
        yanchor="bottom",
        textangle=-80,
    )

fig.update_layout(
    plot_bgcolor="white",
    barmode="stack",
    font=dict(family="Arial", size=12),
    title=dict(
        text=(
            "<b>Age rank distribution by gene family size - Expressed transcripts</b>"
            "<br><sup>Gene family size defined at genome level · "
            "color = age rank (brown = ancient, green = recent)</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    xaxis=dict(
        title="Gene family size (genome)",
        tickmode="linear",
        dtick=5,
        showgrid=False,
    ),
    yaxis=dict(
        title="Proportion of transcripts (%)",
        showgrid=True,
        gridcolor="lightgrey",
        zeroline=False,
        range=[0, 110],
        ticksuffix="%",
    ),
    legend=dict(
        title="Age rank",
        orientation="h",
        yanchor="bottom", y=1.02,
        xanchor="right", x=1,
    ),
    margin=dict(b=80),
    width=1200,
    height=600,
)
fig.show()

All gene family sizes are dominated by duplications in C_mac (except for singeltons which only have one copy in C_mac). A majority of all gene family expansions have occured within C_mac. 

In [81]:
from plotly.subplots import make_subplots

interact_df = de_results_hog[
    de_results_hog["age_rank"].notna() &
    de_results_hog["hog_size_genome"].notna()
].copy()
interact_df["age_rank"]  = interact_df["age_rank"].astype(int)
interact_df["age_label"] = interact_df["age_rank"].map(AGE_LABEL_MAP)
interact_df["size_bin"]  = interact_df["hog_size_genome"].apply(size_bin)

all_labels = [AGE_LABEL_MAP[i] for i in range(1, 10)]

fig = make_subplots(
    rows=1, cols=4,
    shared_yaxes=True,
)

for col_idx, bin_cat in enumerate(bin_order, start=1):
    bin_df = interact_df[interact_df["size_bin"] == bin_cat]

    counts = (
        bin_df.groupby(["age_rank", "age_label", "bias"])
        .size().reset_index(name="count")
    )
    totals = counts.groupby("age_rank")["count"].sum().reset_index(name="total")
    counts = counts.merge(totals, on="age_rank")
    counts["pct"] = counts["count"] / counts["total"] * 100

    for bias, color in BIAS_COLORS.items():
        sub = counts[counts["bias"] == bias].sort_values("age_rank")

        props, texts, customdata = [], [], []
        for rank in range(1, 10):
            row = sub[sub["age_rank"] == rank]
            if len(row) > 0:
                p = row.iloc[0]["pct"]
                n = row.iloc[0]["count"]
                t = row.iloc[0]["total"]
                props.append(p)
                texts.append(f"{p:.1f}%" if p >= 6 else "")
                customdata.append([n, t])
            else:
                props.append(0)
                texts.append("")
                customdata.append([0, 0])

        fig.add_trace(
            go.Bar(
                x=all_labels,
                y=props,
                name=bias,
                marker_color=color,
                marker_line_color="black",
                marker_line_width=0.5,
                text=texts,
                textposition="inside",
                textfont=dict(size=8, color="white"),
                customdata=customdata,
                showlegend=(col_idx == 1),
                hovertemplate=(
                    f"<b>{bias}</b><br>"
                    "Age rank: %{x}<br>"
                    "Proportion: %{y:.1f}%<br>"
                    "Count: %{customdata[0]}<br>"
                    "Total at this rank: %{customdata[1]}<extra></extra>"
                ),
            ),
            row=1, col=col_idx
        )

    # n annotations per facet
    for rank in range(1, 10):
        rank_rows = counts[counts["age_rank"] == rank]
        if len(rank_rows) > 0:
            total = rank_rows["total"].iloc[0]
            fig.add_annotation(
                x=AGE_LABEL_MAP[rank],
                y=1,
                yref=f"y{col_idx if col_idx > 1 else ''} domain",
                text=f"(n={total})",
                showarrow=False,
                yshift=-25,
                font=dict(size=10, color="black"),
                textangle=-80,
                yanchor="bottom",
                row=1, col=col_idx
            )

    # x-axis title below each subplot
    fig.update_xaxes(
        title=dict(text=bin_cat, font=dict(size=16)),
        tickangle=-45,
        tickfont=dict(size=8),
        showgrid=False,
        categoryorder="array",
        categoryarray=all_labels,
        row=1, col=col_idx
    )

fig.update_layout(
    plot_bgcolor="white",
    barmode="stack",
    font=dict(family="Arial", size=11),
    title=dict(
        text=(
            "<b>Sex bias proportions by age rank - subsetted by gene family size</b>"
            "<br><sup>Gene family size at genome level · "
            "Sex bias: padj < 0.05, |LFC| > 1</sup>"
        ),
        x=0.5, xanchor="center"
    ),
    legend=dict(
        title="Sex bias",
        orientation="h",
        yanchor="bottom", y=1.08,
        xanchor="right", x=1,
    ),
    yaxis=dict(
        title="Proportion of transcripts (%)",
        range=[0, 110],
        ticksuffix="%",
        showgrid=True,
        gridcolor="lightgrey",
    ),
    margin=dict(b=80, t=120),
    width=1200,
    height=500,
)

fig.show()